# Quem quer mudar a proibição das bets? As 99 emendas à MP 1.394

A **MP 1.394/2026** (25/set/2026) proíbe as apostas de quota fixa: apostas esportivas **e** cassino online. Ela ainda
**não foi votada**: sem votação, não existe lista de quem votou contra. O que existe são as **emendas** que deputados e
senadores apresentaram na comissão mista. Elas mostram o que cada parlamentar quer mudar no texto.

**Dados:** `uv run python -m src.politica.emendas_mp1394` baixa as emendas dos dados abertos do Senado (autor, partido, UF
e o PDF de cada uma). A classificação está em `src/politica/classificacao_mp1394.csv`, com um resumo de uma linha por emenda.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

import pandas as pd
import polars as pl
from IPython.display import HTML, display

from src.config import pasta

pd.set_option("display.max_colwidth", 120)
CATEGORIAS = {
    1: "Derruba ou esvazia a proibição",
    2: "Proíbe só o cassino online e libera a aposta esportiva",
    3: "Mantém o cassino online, com regras",
    4: "Ameniza para empresas e esporte",
    5: "Regras que pressupõem o mercado funcionando",
    6: "Endurece a proibição",
    7: "Salvaguardas caso alguma aposta seja readmitida",
    8: "Saúde, assistência e destino dos recursos",
    9: "Técnica ou outros",
}
POSICAO = {1: "afrouxa", 2: "afrouxa", 3: "afrouxa", 4: "afrouxa", 5: "afrouxa", 6: "reforça", 7: "reforça", 8: "neutra", 9: "neutra"}
em = pl.read_parquet(pasta("congresso") / "mp1394_emendas.parquet").drop("texto")
cl = pl.read_csv(ROOT / "src" / "politica" / "classificacao_mp1394.csv")
d = em.join(cl, on="numero").with_columns(
    pl.col("categoria").replace_strict(CATEGORIAS).alias("tipo"),
    pl.col("categoria").replace_strict(POSICAO).alias("posição"),
    (pl.col("parlamentar") + " (" + pl.col("partido") + "/" + pl.col("uf") + ")").alias("autor"))
print(f"{d.height} emendas, {d['autor'].n_unique()} autores, apresentadas de {d['data'].min()} a {d['data'].max()}")

99 emendas, 25 autores, apresentadas de 2026-09-29 a 2026-10-13


## Como as emendas foram classificadas

Cada emenda foi **lida** (texto do PDF: o que muda na MP e a justificativa) e colocada num de nove tipos, pela relação com a
proibição:

| Tipo | O que a emenda faz | Posição diante da MP |
|---|---|---|
| 1. Derruba ou esvazia | Mantém as empresas autorizadas operando, suprime os artigos da proibição ou restabelece as autorizações | afrouxa |
| 2. Só cassino online | Troca a proibição total por uma proibição só de caça-níqueis e cassino online ("tigrinho"), liberando as apostas esportivas | afrouxa |
| 3. Mantém o cassino | Deixa o cassino online funcionar com regras (design seguro, publicidade em horário restrito, avaliação antes de proibir) | afrouxa |
| 4. Ameniza | Devolução de outorga, anistia, transição para patrocínios, exceções (derivativos, estados), sanções mais brandas a plataformas | afrouxa |
| 5. Pressupõe o mercado | Regras de jogo responsável, design ou arrecadação que só fazem sentido se as empresas continuarem operando | afrouxa |
| 6. Endurece | Multas maiores, crimes, bloqueio de pagamentos, mais órgãos fiscalizando | reforça |
| 7. Salvaguardas | O autor apoia a proibição e propõe regras rígidas **caso** o Congresso readmita alguma aposta | reforça |
| 8. Saúde e recursos | Tratamento no SUS, apoio a endividados, destino de multas e saldos | neutra |
| 9. Técnica | Ajustes de procedimento | neutra |

**Decisões e limites:**
- A classificação é **manual**, feita lendo o texto, e pode ter erros. Cada linha do CSV tem o resumo que a justifica, e a
  tabela abaixo traz o link do PDF original para conferir.
- O tipo 5 é o mais discutível: regras de proteção ao apostador são boas em si, mas só se aplicam se a proibição cair.
  Por isso entram como "afrouxa". O tipo 7 é o oposto: regras para o caso de readmissão, de autores que declaram apoiar
  a proibição.
- **Emenda não é voto.** Um parlamentar pode apresentar emendas que não votaria, apresentar a pedido de um setor, ou
  apresentar alternativas para negociar. Também há emendas idênticas de autores diferentes, o que costuma indicar texto
  preparado por terceiros (por exemplo, 12, 14, 16 e 20; ou 34, 35, 39 e 40).
- Só 25 dos 594 parlamentares apresentaram emendas. Quem não apresentou não tem posição registrada aqui.

## 1. Quantas emendas de cada tipo

In [2]:
t = d.group_by("categoria", "tipo", "posição").len("emendas").sort("categoria")
t.drop("categoria").to_pandas().set_index("tipo")

,posição,emendas
tipo,,
Derruba ou esvazia a proibição,afrouxa,12
Proíbe só o cassino online e libera a aposta esportiva,afrouxa,8
"Mantém o cassino online, com regras",afrouxa,4
Ameniza para empresas e esporte,afrouxa,17
Regras que pressupõem o mercado funcionando,afrouxa,9
Endurece a proibição,reforça,17
Salvaguardas caso alguma aposta seja readmitida,reforça,13
"Saúde, assistência e destino dos recursos",neutra,14
Técnica ou outros,neutra,5


## 2. Cada parlamentar: o que propôs

In [3]:
por_autor = (d.group_by("autor", "cargo", "posição").len().pivot(on="posição", index=["autor", "cargo"], values="len").fill_null(0))
for c in ["afrouxa", "reforça", "neutra"]:
    if c not in por_autor.columns:
        por_autor = por_autor.with_columns(pl.lit(0).alias(c))
por_autor = por_autor.with_columns((pl.col("afrouxa") + pl.col("reforça") + pl.col("neutra")).alias("total"),
    pl.when((pl.col("afrouxa") > 0) & (pl.col("reforça") == 0)).then(pl.lit("só afrouxa"))
      .when((pl.col("reforça") > 0) & (pl.col("afrouxa") == 0)).then(pl.lit("só reforça"))
      .when((pl.col("afrouxa") > 0) & (pl.col("reforça") > 0)).then(pl.lit("as duas coisas"))
      .otherwise(pl.lit("só saúde/técnica")).alias("resumo"))
por_autor.select("autor", "cargo", "afrouxa", "reforça", "neutra", "total", "resumo").sort(["resumo", "afrouxa"], descending=[False, True]).to_pandas().set_index("autor")

,cargo,afrouxa,reforça,neutra,total,resumo
autor,,,,,,
Marcos Braz (PSDB/RJ),Deputado(a),3,1,1,5,as duas coisas
Capitão Alden (PL/BA),Deputado(a),15,0,2,17,só afrouxa
Julio Arcoverde (PP/PI),Deputado(a),6,0,0,6,só afrouxa
Bebeto (sem partido/RJ),Deputado(a),5,0,0,5,só afrouxa
Mauricio Neves (PP/SP),Deputado(a),4,0,0,4,só afrouxa
Rodrigo Valadares (PL/SE),Deputado(a),4,0,0,4,só afrouxa
Jorge Kajuru (PSB/GO),Senador(a),3,0,0,3,só afrouxa
Átila Lira (PP/PI),Deputado(a),3,0,0,3,só afrouxa
Pedro Aihara (PP/MG),Deputado(a),2,0,0,2,só afrouxa


## 3. Quem propôs derrubar ou esvaziar a proibição (tipo 1)

In [4]:
def lista(cat):
    x = d.filter(pl.col("categoria") == cat).sort("numero").select("numero", "autor", "resumo", "url").to_pandas()
    x["emenda"] = x.apply(lambda r: f'<a href="{r.url}" target="_blank">nº {r.numero}</a>', axis=1)
    return HTML(x[["emenda", "autor", "resumo"]].to_html(escape=False, index=False))

lista(1)

emenda,autor,resumo
nº 28,Julio Arcoverde (PP/PI),Proibição vale só para quem não tem autorização; operadores autorizados continuam
nº 43,Bebeto (sem partido/RJ),MP passa a valer só contra operadores não autorizados
nº 45,Bebeto (sem partido/RJ),Publicidade proibida só para empresas não autorizadas
nº 46,Bebeto (sem partido/RJ),Publicidade proibida só para empresas não autorizadas
nº 51,Capitão Alden (PL/BA),Proibição vale só para quem não tem autorização; operadores autorizados continuam
nº 52,Capitão Alden (PL/BA),Suprime os capítulos que proíbem as apostas
nº 53,Capitão Alden (PL/BA),Proíbe só apostas não autorizadas
nº 59,Bebeto (sem partido/RJ),Suprime os artigos que proíbem as apostas
nº 64,Mauricio Neves (PP/SP),"Reescreve a MP: em vez de proibir, regula publicidade e design e restabelece o mercado autorizado"
nº 69,Capitão Alden (PL/BA),Autorizações continuam válidas até o fim do prazo original


## 4. Quem propôs proibir só o cassino online e liberar a aposta esportiva (tipo 2)

É a alternativa mais apresentada à proibição total: tirar o "tigrinho" e manter as apostas em futebol e outros esportes.

In [5]:
lista(2)

emenda,autor,resumo
nº 10,Marcos Braz (PSDB/RJ),Proíbe só caça-níqueis/cassinos online e apostas em lances isolados; mantém apostas esportivas
nº 12,Pedro Aihara (PP/MG),"Proíbe só o ""tigrinho"" e jogos de slot; mantém apostas esportivas"
nº 14,Pedro Aihara (PP/MG),"Proíbe só o ""tigrinho"" e jogos de slot; mantém apostas esportivas"
nº 15,Astronauta Marcos Pontes (PL/SP),Proíbe só os jogos online; apostas esportivas continuam regidas pela Lei 14.790
nº 16,Julio Lopes (PP/RJ),"Proíbe só o ""tigrinho"" e jogos de slot; mantém apostas esportivas"
nº 20,Jorge Kajuru (PSB/GO),"Proíbe só o ""tigrinho"" e jogos de slot; mantém apostas esportivas"
nº 38,Dr. Hiran (PP/RR),Proíbe jogos online; restringe mas mantém apostas em eventos esportivos reais
nº 55,Romário (sem partido/RJ),Proíbe jogos online e apostas em lances; mantém apostas no resultado final de jogos esportivos


## 5. Quem propôs manter o cassino online (tipo 3)

In [6]:
lista(3)

emenda,autor,resumo
nº 67,Capitão Alden (PL/BA),Exige avaliação de impacto de 12 meses antes de proibir jogos online
nº 70,Capitão Alden (PL/BA),Governo poderá restringir jogos online específicos em vez de proibir a modalidade
nº 72,Capitão Alden (PL/BA),Permite publicidade de jogos online por operadores autorizados (entre 21h e 6h na TV)
nº 73,Capitão Alden (PL/BA),Regime de design seguro para jogos online de operadores autorizados


## 6. Quem propôs endurecer a proibição (tipos 6 e 7)

In [7]:
x = d.filter(pl.col("categoria").is_in([6, 7])).group_by("autor").agg(pl.len().alias("emendas"), pl.col("resumo").head(3).str.join(" · ").alias("exemplos")).sort("emendas", descending=True)
x.to_pandas().set_index("autor")

,emendas,exemplos
autor,,
Sargento Reginauro (PSDB/CE),16,Se readmitidas: veda práticas de design que induzem a apostar mais · Se readmitidas: 2% da arrecadação para o Fundo ...
Alessandro Vieira (MDB/SE),5,Processos contra lavagem de dinheiro e apostas de menores não são suspensos · Prazos para o Banco Central bloquear p...
Beto Preto (PSD/PR),4,Eleva a multa diária por descumprimento para R$ 500 mil · Comitê de fiscalização coordenado pelo Ministério da Justi...
Hildo Rocha (MDB/MA),3,Comitê de fiscalização coordenado pelo Ministério da Justiça · Multa a plataformas sobe de 10% para 50% do faturamen...
Marcos Braz (PSDB/RJ),1,Proíbe apostas por beneficiários de programas de renda (Bolsa Família etc.)
Julio Cesar Ribeiro (REPUBLICANOS/DF),1,Proibição de pagamentos alcança intermediários e contas de terceiros


## 7. Por partido

In [8]:
pp = d.group_by("partido", "posição").len().pivot(on="posição", index="partido", values="len").fill_null(0)
pp.with_columns(pl.sum_horizontal(pl.exclude("partido")).alias("total")).sort("total", descending=True).to_pandas().set_index("partido")

,neutra,reforça,afrouxa,total
partido,,,,
PL,4,0,20,24
PSDB,2,17,3,22
PP,0,0,18,18
MDB,4,8,0,12
sem partido,0,0,6,6
PSD,1,4,0,5
PSB,1,0,3,4
REPUBLICANOS,2,1,0,3
PT,3,0,0,3


⚠ A contagem por partido mede **quantas emendas**, não quantos parlamentares. Um autor muito ativo pesa muito: o
dep. Capitão Alden (PL/BA) sozinho apresentou 17 emendas, e o sen. Sargento Reginauro (PSDB/CE), outras 17.

## Leitura

**1. A maioria das emendas afrouxa a proibição.** Cerca de metade (50 de 99) vai nessa direção: 12 derrubam ou esvaziam
a MP, 8 trocam a proibição total por uma proibição só do cassino online, 4 mantêm o cassino com regras, 17 amenizam a
transição para as empresas e o esporte, e 9 criam regras que só fazem sentido com o mercado funcionando. Outras 30
reforçam a proibição, e 19 tratam de saúde, destino dos recursos ou ajustes técnicos.

**2. Poucos autores concentram a pressão por afrouxar.** O dep. **Capitão Alden (PL/BA)** apresentou 15 emendas que afrouxam,
de todos os tipos: derrubar a MP, devolver as outorgas, manter o cassino online, liberar publicidade. Também aparecem com
propostas de derrubar ou esvaziar o dep. **Bebeto (sem partido/RJ)**, o dep. **Julio Arcoverde (PP/PI)**, o dep.
**Mauricio Neves (PP/SP)** e o dep. **Átila Lira (PP/PI)**. O PP aparece com vários autores nessa linha.

**3. A alternativa mais apresentada é "proibir só o tigrinho".** Oito emendas, de autores de partidos diferentes (PP, PL,
PSB, PSDB e Romário, sem partido), propõem acabar com o cassino online e manter as apostas esportivas. Quatro delas têm
texto praticamente idêntico (12, 14, 16 e 20). É a linha de negociação mais provável se a MP for mudada no Congresso.

**4. Quem quer endurecer também está no Congresso.** O sen. **Sargento Reginauro (PSDB/CE)** apresentou 16 emendas para
reforçar a proibição, várias delas regras rígidas "caso o Congresso readmita alguma aposta", e uma (86) para que o cassino
online **nunca** possa ser readmitido. O sen. **Alessandro Vieira (MDB/SE)** propôs criar crimes, responsabilizar bancos
e destinar multas ao SUS. O dep. **Beto Preto (PSD/PR)** e o dep. **Hildo Rocha (MDB/MA)** propuseram multas maiores.

**5. A saúde aparece pouco, mas aparece.** As deps. **Tabata Amaral (PSB/SP)** e **Ana Paula Lima (PT/SC)** propuseram uma
linha de cuidado no SUS para o transtorno do jogo, com os CAPS e a atenção primária, exatamente a rede em que os dados
deste projeto mostram a alta de atendimentos. Várias emendas mandam multas e saldos apreendidos para o Fundo Nacional
de Saúde.

**O que isso não mostra:** como cada parlamentar vai votar. A posição real só aparece na votação da comissão mista e
dos plenários. A MP perde a validade se não for votada até mar/2027.